In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sklearn.cluster import KMeans
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
from sklearn.preprocessing import StandardScaler
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D1 in response_OUS
data = list(OUS_D1['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D1, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [5]:
# Check null values in D1
clinical_train.isnull().sum().sum()

0

## Test dataset: MAASTRO 

In [6]:
(MAASTRO_D1['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [7]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [8]:
# need to choose patient_id from MAASTRO_D1 in response_MAASTRO
data = list(MAASTRO_D1['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 

In [9]:
# Merge MAASTRO_D2 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D1, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'DFS_event', 'LRC', 'LRC_event'])]

In [10]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [11]:
# Check if some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,OS,OS_event


In [12]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS'])]

# y 
y = clinical_train.loc[:, ['OS', 'event_OS']]

In [13]:
# Set lower, upper time point and times for IBS calculation later 

# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

clinical_test.rename(columns = {'OS_event' : 'event_OS'}, inplace = True)

# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'event_OS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['OS', 'event_OS']]
lower, upper = np.percentile(y_MAASTRO['OS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_OS'], y_MAASTRO['OS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

X_train:  (139, 14)
y_train:  (139,)


(99, 16)

# Feature Selection: PLSR

In [14]:
selected_features = [ "hpv_related",
"oropharynx",
"uicc8_III-IV",
"cavum_oris",
"MTV",
"TLG",
"charlson",
"age" ]

# Selecting features in the DataFrame
X_rent = X[selected_features]

In [15]:
# Selecting features in the DataFrame
X_rent = X[selected_features]
X_new = X_rent.copy()

X_MAASTRO_rent = X_MAASTRO[selected_features]
MAASTRO_new = X_MAASTRO_rent.copy()

# Standardization

In [16]:
# Standardize X_new, the new data with the selected features only 
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
scaler = StandardScaler() 
X_new_numeric_columns = X_new_numeric.columns
X_new_numeric_index = X_new_numeric.index 
X_new_numeric_std = scaler.fit_transform(X_new_numeric)
X_new_numeric_std = pd.DataFrame(X_new_numeric_std,
                                 columns=X_new_numeric_columns, 
                                 index=X_new_numeric_index)
X_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]

In [19]:
# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns

MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]
MAASTRO_new = MAASTRO_new[X_new.columns]
MAASTRO_new_std = MAASTRO_new_std[X_new.columns]

In [20]:
X_new

,hpv_related,oropharynx,uicc8_III-IV,cavum_oris,MTV,TLG,charlson,age
0,0.0,1,0.0,0,7.934,86.228420,0,54.238356
1,0.0,0,0.0,0,1.656,7.040100,1,54.539726
2,0.0,0,1.0,1,14.502,83.569669,1,59.019178
3,0.0,0,0.0,0,2.440,5.567091,1,70.726027
4,0.0,0,0.0,0,3.668,16.150550,1,67.865753
...,...,...,...,...,...,...,...,...
134,1.0,1,0.0,0,3.650,26.280140,0,60.435616
135,1.0,1,1.0,0,18.967,101.754834,0,68.794521
136,1.0,1,0.0,0,6.370,66.273201,1,57.498630
137,1.0,1,1.0,0,12.443,71.832443,1,65.684932


In [21]:
X_new_std

,hpv_related,oropharynx,uicc8_III-IV,cavum_oris,MTV,TLG,charlson,age
0,0.0,1,0.0,0,-0.296907,-0.179158,0,-0.776210
1,0.0,0,0.0,0,-0.763433,-0.587333,1,-0.737256
2,0.0,0,1.0,1,0.191169,-0.192863,1,-0.158251
3,0.0,0,0.0,0,-0.705173,-0.594926,1,1.354953
4,0.0,0,0.0,0,-0.613919,-0.540373,1,0.985240
...,...,...,...,...,...,...,...,...
134,1.0,1,0.0,0,-0.615257,-0.488161,0,0.024835
135,1.0,1,1.0,0,0.522969,-0.099128,0,1.105290
136,1.0,1,0.0,0,-0.413130,-0.282017,1,-0.354794
137,1.0,1,1.0,0,0.038162,-0.253362,1,0.703351


In [22]:
MAASTRO_new

,hpv_related,oropharynx,uicc8_III-IV,cavum_oris,MTV,TLG,charlson,age
0,1,1,0,0,22.841,263.611623,1,55
1,0,1,1,0,5.660,36.980700,0,55
2,0,1,1,0,7.791,74.636342,1,55
3,0,0,1,0,7.908,46.791979,1,61
4,1,1,0,0,15.237,107.637514,1,70
...,...,...,...,...,...,...,...,...
94,0,0,1,0,6.110,144.490782,0,66
95,0,0,1,0,7.182,69.214868,1,63
96,1,1,1,0,16.483,102.594274,1,63
97,1,1,0,0,9.981,103.229492,0,54


In [23]:
MAASTRO_new_std

,hpv_related,oropharynx,uicc8_III-IV,cavum_oris,MTV,TLG,charlson,age
0,1,1,0,0,0.810851,0.735160,1,-0.677762
1,0,1,1,0,-0.465891,-0.433005,0,-0.677762
2,0,1,1,0,-0.307534,-0.238909,1,-0.677762
3,0,0,1,0,-0.298839,-0.382433,1,0.097786
4,1,1,0,0,0.245788,-0.068806,1,1.261108
...,...,...,...,...,...,...,...,...
94,0,0,1,0,-0.432451,0.121154,0,0.744076
95,0,0,1,0,-0.352789,-0.266854,1,0.356302
96,1,1,1,0,0.338380,-0.094801,1,0.356302
97,1,1,0,0,-0.144792,-0.091527,0,-0.807020


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [25]:
# Setting the y format for skf below  
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 23:50:51,493] A new study created in memory with name: no-name-dbdd796a-d1db-4b76-bd28-b3019e00e21a


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.7316017316017316
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8481012658227848


[I 2024-04-13 23:50:52,485] A new study created in memory with name: no-name-74f33ede-7a79-4c5d-9558-e192f351efb7


Fold 5 C-index: 0.6431924882629108
[I 2024-04-13 23:50:52,439] Trial 0 finished with value: 0.7580419822915472 and parameters: {}. Best is trial 0 with value: 0.7580419822915472.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7580419822915472], datetime_start=datetime.datetime(2024, 4, 13, 23, 50, 51, 609787), datetime_complete=datetime.datetime(2024, 4, 13, 23, 50, 52, 438223), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7580419822915472


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.1841011974655529
Fold 2 IBS: 0.16232564718800943
Fold 3 IBS: 0.16363945158036763
Fold 4 IBS: 0.1454766840436639
Fold 5 IBS: 0.25454696204754557
[I 2024-04-13 23:50:53,672] Trial 0 finished with value: 0.18201798846502787 and parameters: {}. Best is trial 0 with value: 0.18201798846502787.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.18201798846502787], datetime_start=datetime.datetime(2024, 4, 13, 23, 50, 52, 751330), datetime_complete=datetime.datetime(2024, 4, 13, 23, 50, 53, 671254), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.18201798846502787


In [26]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [27]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.758
train_ibs:  0.182


#### Test

In [28]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [29]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.621
IBS score: 0.261


In [30]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [31]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge 

#### Train

In [32]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 23:50:56,033] A new study created in memory with name: no-name-704c7274-08b6-4823-a3ec-d2c537021e92


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5735930735930735
Fold 2 C-index: 0.6026785714285714
Fold 3 C-index: 0.7352941176470589


[I 2024-04-13 23:50:56,542] A new study created in memory with name: no-name-92050c85-4218-48f2-9887-0a3bf81782a4


Fold 4 C-index: 0.7911392405063291
Fold 5 C-index: 0.6901408450704225
[I 2024-04-13 23:50:56,523] Trial 0 finished with value: 0.678569169649091 and parameters: {}. Best is trial 0 with value: 0.678569169649091.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.678569169649091], datetime_start=datetime.datetime(2024, 4, 13, 23, 50, 56, 72557), datetime_complete=datetime.datetime(2024, 4, 13, 23, 50, 56, 523394), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.678569169649091


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.21397651907207446
Fold 2 IBS: 0.221577912661353
Fold 3 IBS: 0.20453594275105988
Fold 4 IBS: 0.22473802858598244
Fold 5 IBS: 0.2181243108260825
[I 2024-04-13 23:50:57,278] Trial 0 finished with value: 0.21659054277931045 and parameters: {}. Best is trial 0 with value: 0.21659054277931045.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.21659054277931045], datetime_start=datetime.datetime(2024, 4, 13, 23, 50, 56, 688452), datetime_complete=datetime.datetime(2024, 4, 13, 23, 50, 57, 278492), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.21659054277931045


In [33]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [34]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.679
train_ibs:  0.217


#### Test

In [35]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [36]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.647


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.221


In [37]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [38]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 23:50:57,864] A new study created in memory with name: no-name-c2b5a404-1880-4265-b4b4-8af68f51eade


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.7316017316017316
Fold 2 C-index: 0.7723214285714286
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8523206751054853
Fold 5 C-index: 0.647887323943662
[I 2024-04-13 23:50:59,815] Trial 0 finished with value: 0.7616105455699517 and parameters: {}. Best is trial 0 with value: 0.7616105455699517.


[I 2024-04-13 23:50:59,947] A new study created in memory with name: no-name-129a32f8-51b9-4865-b85b-25ed2de8b07d




* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7616105455699517], datetime_start=datetime.datetime(2024, 4, 13, 23, 50, 57, 928788), datetime_complete=datetime.datetime(2024, 4, 13, 23, 50, 59, 815170), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7616105455699517


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.18447793741411198
Fold 2 IBS: 0.16484848693447857
Fold 3 IBS: 0.16251969838744912
Fold 4 IBS: 0.14389244536199547
Fold 5 IBS: 0.25277413959278777
[I 2024-04-13 23:51:01,726] Trial 0 finished with value: 0.18170254153816456 and parameters: {}. Best is trial 0 with value: 0.18170254153816456.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.18170254153816456], datetime_start=datetime.datetime(2024, 4, 13, 23, 50, 59, 986605), datetime_complete=datetime.datetime(2024, 4, 13, 23, 51, 1, 725712), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.18170254153816456


In [39]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [40]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.762
train_ibs:  0.182


#### Test 

In [41]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [42]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.623


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.258


In [43]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [44]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 23:51:02,957] A new study created in memory with name: no-name-a0b9c934-78e3-428d-941b-b18016f3e466


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7316017316017316
Fold 2 C-index: 0.7678571428571429
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8523206751054853
Fold 5 C-index: 0.647887323943662
[I 2024-04-13 23:51:04,494] Trial 0 finished with value: 0.7607176884270945 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.7607176884270945.
Fold 1 C-index: 0.7316017316017316
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8523206751054853
Fold 5 C-index: 0.6431924882629108
[I 2024-04-13 23:51:05,826] Trial 1 finished with value: 0.7588858641480872 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.7607176884270945.
Fold 1 C-index: 0.7316017316017316
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8523206751054853
Fold 5 C-index: 0.6431924882629108
[I 2024-04-13 23:51:07,053] Trial 2 finished with value: 0.7588858641480872 and parameters: {'l1_ratio': 0.22692876841884668}. Bes

Fold 1 C-index: 0.7316017316017316
Fold 2 C-index: 0.7723214285714286
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8523206751054853
Fold 5 C-index: 0.647887323943662
[I 2024-04-13 23:51:46,607] Trial 24 finished with value: 0.7616105455699517 and parameters: {'l1_ratio': 0.9954177043436652}. Best is trial 6 with value: 0.7616105455699517.
Fold 1 C-index: 0.7316017316017316
Fold 2 C-index: 0.7723214285714286
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8523206751054853
Fold 5 C-index: 0.647887323943662
[I 2024-04-13 23:51:48,031] Trial 25 finished with value: 0.7616105455699517 and parameters: {'l1_ratio': 0.891573855041803}. Best is trial 6 with value: 0.7616105455699517.
Fold 1 C-index: 0.7316017316017316
Fold 2 C-index: 0.7723214285714286
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8523206751054853
Fold 5 C-index: 0.647887323943662
[I 2024-04-13 23:51:49,474] Trial 26 finished with value: 0.7616105455699517 and parameters: {'l1_ratio': 0.6628070200466962}. Best 

Fold 1 C-index: 0.7316017316017316
Fold 2 C-index: 0.7723214285714286
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8523206751054853
Fold 5 C-index: 0.647887323943662
[I 2024-04-13 23:52:22,985] Trial 48 finished with value: 0.7616105455699517 and parameters: {'l1_ratio': 0.9468743846989455}. Best is trial 6 with value: 0.7616105455699517.
Fold 1 C-index: 0.7316017316017316
Fold 2 C-index: 0.7723214285714286
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8523206751054853
Fold 5 C-index: 0.647887323943662
[I 2024-04-13 23:52:24,457] Trial 49 finished with value: 0.7616105455699517 and parameters: {'l1_ratio': 0.6437567638801116}. Best is trial 6 with value: 0.7616105455699517.
Fold 1 C-index: 0.7316017316017316
Fold 2 C-index: 0.7678571428571429
Fold 3 C-index: 0.7990196078431373
Fold 4 C-index: 0.8523206751054853
Fold 5 C-index: 0.647887323943662
[I 2024-04-13 23:52:25,995] Trial 50 finished with value: 0.7597372962702318 and parameters: {'l1_ratio': 0.556147470734502}. Best

...

Fold 1 IBS: 0.18453114438656854
Fold 2 IBS: 0.16524846540781152
Fold 3 IBS: 0.16240371602679218
Fold 4 IBS: 0.14396719787048534
Fold 5 IBS: 0.25249548312425735
[I 2024-04-13 23:53:42,802] Trial 0 finished with value: 0.181729201363183 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.181729201363183.
Fold 1 IBS: 0.18468460851287097
Fold 2 IBS: 0.1654620040344674
Fold 3 IBS: 0.16229687195493578
Fold 4 IBS: 0.14397721472972333
Fold 5 IBS: 0.25220067836759713
[I 2024-04-13 23:53:47,032] Trial 1 finished with value: 0.18172427551991893 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.18172427551991893.
Fold 1 IBS: 0.18470448380172774
Fold 2 IBS: 0.16529155765895748
Fold 3 IBS: 0.16221726179755092
Fold 4 IBS: 0.1439337100543988
Fold 5 IBS: 0.25205159057623044
[I 2024-04-13 23:53:49,216] Trial 2 finished with value: 0.18163972077777307 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.181639720777773

Fold 1 IBS: 0.1846419084746135
Fold 2 IBS: 0.1652944985172605
Fold 3 IBS: 0.16228874983173466
Fold 4 IBS: 0.14392367586563456
Fold 5 IBS: 0.25220359364742023
[I 2024-04-13 23:54:58,056] Trial 25 finished with value: 0.1816704852673327 and parameters: {'l1_ratio': 0.36669581947150787}. Best is trial 14 with value: 0.18162870450635601.
Fold 1 IBS: 0.18469363577411685
Fold 2 IBS: 0.1655319352864163
Fold 3 IBS: 0.16223724598073103
Fold 4 IBS: 0.1439481123428458
Fold 5 IBS: 0.2521007582494757
[I 2024-04-13 23:55:00,169] Trial 26 finished with value: 0.18170233752671713 and parameters: {'l1_ratio': 0.2526650212752227}. Best is trial 14 with value: 0.18162870450635601.
Fold 1 IBS: 0.1846028666869189
Fold 2 IBS: 0.16546047982658618
Fold 3 IBS: 0.16228397756521096
Fold 4 IBS: 0.14398240301244816
Fold 5 IBS: 0.2522301018918221
[I 2024-04-13 23:55:03,820] Trial 27 finished with value: 0.18171196579659724 and parameters: {'l1_ratio': 0.4698787873988891}. Best is trial 14 with value: 0.181628704506

Fold 5 IBS: 0.25230593732216156
[I 2024-04-13 23:55:58,762] Trial 49 finished with value: 0.18170123094856183 and parameters: {'l1_ratio': 0.4477016529538363}. Best is trial 30 with value: 0.18161020234777872.
Fold 1 IBS: 0.18470245288853446
Fold 2 IBS: 0.16542012670485762
Fold 3 IBS: 0.16223225976599265
Fold 4 IBS: 0.14398376501312177
Fold 5 IBS: 0.2520586853382567
[I 2024-04-13 23:56:02,356] Trial 50 finished with value: 0.18167945794215265 and parameters: {'l1_ratio': 0.16135214111227222}. Best is trial 30 with value: 0.18161020234777872.
Fold 1 IBS: 0.18478452362497247
Fold 2 IBS: 0.16533923850932122
Fold 3 IBS: 0.16216272631119075
Fold 4 IBS: 0.14397924340216348
Fold 5 IBS: 0.2519158103076707
[I 2024-04-13 23:56:07,038] Trial 51 finished with value: 0.1816363084310637 and parameters: {'l1_ratio': 0.047112746267578}. Best is trial 30 with value: 0.18161020234777872.
Fold 1 IBS: 0.18477217738020543
Fold 2 IBS: 0.16534260435166545
Fold 3 IBS: 0.16216796460414784
Fold 4 IBS: 0.1439794

Fold 1 IBS: 0.1846902594757335
Fold 2 IBS: 0.16536597636556824
Fold 3 IBS: 0.16225902288308378
Fold 4 IBS: 0.14388690304010943
Fold 5 IBS: 0.25210777682637997
[I 2024-04-13 23:57:03,682] Trial 74 finished with value: 0.181661987718175 and parameters: {'l1_ratio': 0.16467357394326002}. Best is trial 30 with value: 0.18161020234777872.
Fold 1 IBS: 0.18476230415188769
Fold 2 IBS: 0.16522952890214168
Fold 3 IBS: 0.16220423073517465
Fold 4 IBS: 0.2214956757734565
Fold 5 IBS: 0.2519814159106742
[I 2024-04-13 23:57:05,605] Trial 75 finished with value: 0.19713463109466695 and parameters: {'l1_ratio': 0.027987763370412786}. Best is trial 30 with value: 0.18161020234777872.
Fold 1 IBS: 0.18475088015796817
Fold 2 IBS: 0.1653300916459578
Fold 3 IBS: 0.1622410024981052
Fold 4 IBS: 0.14389969927773982
Fold 5 IBS: 0.2520650150970549
[I 2024-04-13 23:57:07,663] Trial 76 finished with value: 0.18165733773536516 and parameters: {'l1_ratio': 0.11354223262588735}. Best is trial 30 with value: 0.181610202

Fold 5 IBS: 0.25200298551120875
[I 2024-04-13 23:58:00,658] Trial 98 finished with value: 0.18165358318231073 and parameters: {'l1_ratio': 0.12064011750017613}. Best is trial 93 with value: 0.1816052742905947.
Fold 1 IBS: 0.1847590492138749
Fold 2 IBS: 0.16533727031398002
Fold 3 IBS: 0.16219991339788398
Fold 4 IBS: 0.14399871750414447
Fold 5 IBS: 0.2519925543172047
[I 2024-04-13 23:58:03,370] Trial 99 finished with value: 0.18165750094941763 and parameters: {'l1_ratio': 0.09182782178528937}. Best is trial 93 with value: 0.1816052742905947.


* Best trial for IBS: 
 FrozenTrial(number=93, state=TrialState.COMPLETE, values=[0.1816052742905947], datetime_start=datetime.datetime(2024, 4, 13, 23, 57, 45, 336712), datetime_complete=datetime.datetime(2024, 4, 13, 23, 57, 47, 663191), params={'l1_ratio': 0.06055283354521909}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'l1_ratio': FloatDistribution(high=1.0, log=False, low=0.0001, step=None)}, trial_id=93, value=None

In [45]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [46]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.762
train_ibs:  0.182


#### Test

In [47]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [48]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.980766121964777)

test_cindex : 0.623


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.06055283354521909)

test_ibs:  0.257


In [49]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [80]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 12:50:01,821] A new study created in memory with name: no-name-9dd939a2-b21b-4d99-9422-89f710f64d48


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.70995670995671
Fold 2 C-index: 0.6517857142857143
Fold 3 C-index: 0.8284313725490197
Fold 4 C-index: 0.8586497890295358
Fold 5 C-index: 0.6384976525821596
[I 2024-04-14 12:50:06,069] Trial 0 finished with value: 0.7374642476806279 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.7374642476806279.
Fold 1 C-index: 0.696969696969697
Fold 2 C-index: 0.6919642857142857
Fold 3 C-index: 0.7696078431372549
Fold 4 C-index: 0.8649789029535865
Fold 5 C-index: 0.6854460093896714
[I 2024-04-14 12:50:10,169] Trial 1 finished with value: 0.7417933476328991 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 'ma

Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.8258928571428571
Fold 3 C-index: 0.7696078431372549
Fold 4 C-index: 0.8713080168776371
Fold 5 C-index: 0.7300469483568075
[I 2024-04-14 12:50:58,815] Trial 17 finished with value: 0.777033470765249 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 9, 'min_samples_leaf': 11, 'max_depth': 7, 'n_estimators': 77, 'oob_score': True, 'max_samples': 0.8699768148803853, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.2109291215895694, 'warm_start': True}. Best is trial 14 with value: 0.78448790823644.
Fold 1 C-index: 0.696969696969697
Fold 2 C-index: 0.796875
Fold 3 C-index: 0.8259803921568627
Fold 4 C-index: 0.8417721518987342
Fold 5 C-index: 0.6619718309859155
[I 2024-04-14 12:50:59,443] Trial 18 finished with value: 0.7647138144022418 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 15, 'min_samples_leaf': 8, 'max_depth': 2, 'n_estimators': 86, 'oob_score': True, 'max_samples': 0.9862374149206906, 'max_features':

Fold 1 C-index: 0.683982683982684
Fold 2 C-index: 0.8660714285714286
Fold 3 C-index: 0.8137254901960784
Fold 4 C-index: 0.8649789029535865
Fold 5 C-index: 0.7112676056338029
[I 2024-04-14 12:51:16,208] Trial 32 finished with value: 0.7880052222675161 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 6, 'min_samples_leaf': 14, 'max_depth': 13, 'n_estimators': 237, 'oob_score': True, 'max_samples': 0.9241435345198769, 'max_features': None, 'min_weight_fraction_leaf': 0.005704708830120669, 'warm_start': True}. Best is trial 26 with value: 0.7971355772328867.
Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.8616071428571429
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.869198312236287
Fold 5 C-index: 0.7183098591549296
[I 2024-04-14 12:51:18,147] Trial 33 finished with value: 0.7853285377669115 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 12, 'max_depth': 16, 'n_estimators': 263, 'oob_score': True, 'max_samples': 0.7342970632385599

...

Fold 5 IBS: 0.21653408955576556
[I 2024-04-14 12:55:13,533] Trial 31 finished with value: 0.18482261157690327 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 14, 'min_samples_leaf': 8, 'max_depth': 2, 'n_estimators': 450, 'oob_score': False, 'max_samples': 0.8790885825126233, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.10708995040974342}. Best is trial 25 with value: 0.18412145468397245.
Fold 1 IBS: 0.18211270990315442
Fold 2 IBS: 0.21293412693121805
Fold 3 IBS: 0.16877883733430096
Fold 4 IBS: 0.1473017057119311
Fold 5 IBS: 0.2196556092852957
[I 2024-04-14 12:55:17,886] Trial 32 finished with value: 0.18615659783318006 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 17, 'min_samples_leaf': 8, 'max_depth': 5, 'n_estimators': 462, 'oob_score': False, 'max_samples': 0.9478519130685139, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.03442061510351581}. Best is trial 25 with value: 0.18412145468397245.
Fold 1 IBS: 0.18130817749522796
Fold 2 IBS: 0.2076

Fold 1 IBS: 0.17495436191336747
Fold 2 IBS: 0.21642383473182975
Fold 3 IBS: 0.16384945069352883
Fold 4 IBS: 0.1436363081374491
Fold 5 IBS: 0.20972010821870968
[I 2024-04-14 12:56:10,897] Trial 47 finished with value: 0.18171681273897697 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 19, 'min_samples_leaf': 1, 'max_depth': 7, 'n_estimators': 240, 'oob_score': True, 'max_samples': 0.2322220006496939, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.007963153823121693}. Best is trial 45 with value: 0.1815634200211001.
Fold 1 IBS: 0.2140400529445786
Fold 2 IBS: 0.22056189970139348
Fold 3 IBS: 0.20499151963199355
Fold 4 IBS: 0.22458997158755165
Fold 5 IBS: 0.2175710899843072
[I 2024-04-14 12:56:13,886] Trial 48 finished with value: 0.21635090676996488 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 19, 'min_samples_leaf': 1, 'max_depth': 7, 'n_estimators': 233, 'oob_score': True, 'max_samples': 0.21313526563727975, 'max_features': 'log2', 'min_weight_fraction_lea

Fold 5 IBS: 0.20788163499136067
[I 2024-04-14 12:57:00,050] Trial 62 finished with value: 0.18176024441372457 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 19, 'min_samples_leaf': 2, 'max_depth': 8, 'n_estimators': 277, 'oob_score': True, 'max_samples': 0.13095385188884462, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.0017332514726945694}. Best is trial 45 with value: 0.1815634200211001.
Fold 1 IBS: 0.1800536074338711
Fold 2 IBS: 0.20967741302207912
Fold 3 IBS: 0.16725955903309653
Fold 4 IBS: 0.14663083503834592
Fold 5 IBS: 0.21313464934329182
[I 2024-04-14 12:57:02,987] Trial 63 finished with value: 0.1833512127741369 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 19, 'min_samples_leaf': 2, 'max_depth': 7, 'n_estimators': 248, 'oob_score': True, 'max_samples': 0.25162324100280287, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.02363064613843352}. Best is trial 45 with value: 0.1815634200211001.
Fold 1 IBS: 0.21400657323048353
Fold 2 IBS: 0.2206

Fold 1 IBS: 0.21402731898199354
Fold 2 IBS: 0.22070117517287685
Fold 3 IBS: 0.20511548294468487
Fold 4 IBS: 0.2245478065185114
Fold 5 IBS: 0.217888359190842
[I 2024-04-14 12:57:51,560] Trial 78 finished with value: 0.21645602856178173 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 6, 'n_estimators': 204, 'oob_score': True, 'max_samples': 0.2626685610811855, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.014757495457958077}. Best is trial 45 with value: 0.1815634200211001.
Fold 1 IBS: 0.1806026877192447
Fold 2 IBS: 0.20799751875386632
Fold 3 IBS: 0.16811000383356925
Fold 4 IBS: 0.15063751286152252
Fold 5 IBS: 0.21025312145554675
[I 2024-04-14 12:57:53,796] Trial 79 finished with value: 0.1835201689247499 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 17, 'min_samples_leaf': 3, 'max_depth': 9, 'n_estimators': 224, 'oob_score': False, 'max_samples': 0.15862553836501161, 'max_features': 'log2', 'min_weight_fraction_le

Fold 5 IBS: 0.20690344572256106
[I 2024-04-14 12:58:37,438] Trial 93 finished with value: 0.18100540098398804 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 18, 'min_samples_leaf': 2, 'max_depth': 9, 'n_estimators': 244, 'oob_score': True, 'max_samples': 0.17462848601577474, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.022583095812189397}. Best is trial 93 with value: 0.18100540098398804.
Fold 1 IBS: 0.18217933960047325
Fold 2 IBS: 0.19851116575329222
Fold 3 IBS: 0.16573397532261386
Fold 4 IBS: 0.16479739333865168
Fold 5 IBS: 0.2091546360901032
[I 2024-04-14 12:58:39,363] Trial 94 finished with value: 0.18407530202102687 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 18, 'min_samples_leaf': 3, 'max_depth': 9, 'n_estimators': 247, 'oob_score': True, 'max_samples': 0.11867050486176221, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.02004822428701594}. Best is trial 93 with value: 0.18100540098398804.
Fold 1 IBS: 0.18210665570530915
Fold 2 IBS: 0.20

In [81]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [82]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.821
train_ibs:  0.181


#### Test

In [83]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [84]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=20, max_features='auto', max_leaf_nodes=18,
                     max_samples=0.64489763430965, min_samples_leaf=1,
                     min_samples_split=10,
                     min_weight_fraction_leaf=0.0018056666954531422,
                     n_estimators=301, random_state=123, warm_start=True)

test_cindex:  0.683


RandomSurvivalForest(max_depth=9, max_features='log2', max_leaf_nodes=18,
                     max_samples=0.17462848601577474, min_samples_leaf=2,
                     min_samples_split=8,
                     min_weight_fraction_leaf=0.022583095812189397,
                     n_estimators=244, oob_score=True, random_state=123)

test_ibs:  0.201


In [85]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [86]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 12:58:49,890] A new study created in memory with name: no-name-d2d97d66-1d15-412f-b281-c37a52530cc3


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7445887445887446
Fold 2 C-index: 0.7901785714285714
Fold 3 C-index: 0.8578431372549019
Fold 4 C-index: 0.8227848101265823
Fold 5 C-index: 0.6784037558685446
[I 2024-04-14 12:58:50,492] Trial 0 finished with value: 0.7787598038534689 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7787598038534689.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 12:58:51,882] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 1 C-index: 0.7662337662337663
Fold 2 C-index: 0.7790178571428571
Fold 3 C-index: 0.8333333333333334
Fold 4 C-index: 0.8227848101265823
Fold 5 C-index: 0.6596244131455399
[I 2024-04-14 12:59:09,631] Trial 16 finished with value: 0.7721988359964158 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.6535355702555379, 'min_weight_fraction_leaf': 0.1906011147608997}. Best is trial 12 with value: 0.7832355192437597.
Fold 1 C-index: 0.7554112554112554
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.8235294117647058
Fold 4 C-index: 0.8037974683544303
Fold 5 C-index: 0.6267605633802817
[I 2024-04-14 12:59:10,295] Trial 17 finished with value: 0.7590425969249918 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 10, 'min_samples_leaf': 8, 'max_depth': 10, 'n_estimators': 384, 'oob_score': False, 'warm_start': True, 'max_featu

Fold 1 C-index: 0.7489177489177489
Fold 2 C-index: 0.8169642857142857
Fold 3 C-index: 0.8578431372549019
Fold 4 C-index: 0.8354430379746836
Fold 5 C-index: 0.6713615023474179
[I 2024-04-14 12:59:18,642] Trial 31 finished with value: 0.7861059424418075 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 15, 'min_samples_leaf': 6, 'max_depth': 14, 'n_estimators': 463, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.8597320351705271, 'min_weight_fraction_leaf': 0.03653370864131273}. Best is trial 31 with value: 0.7861059424418075.
Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.8169642857142857
Fold 3 C-index: 0.8578431372549019
Fold 4 C-index: 0.8438818565400844
Fold 5 C-index: 0.6643192488262911
[I 2024-04-14 12:59:19,488] Trial 32 finished with value: 0.7837878528532598 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 15, 'min_samples_leaf': 6, 'max_depth': 13, 'n_estimators': 459, 'oob_score': False, 'warm_start': True, 'max_fea

...

Fold 1 IBS: 0.1753265543826254
Fold 2 IBS: 0.2073287496181804
Fold 3 IBS: 0.17214889802085478
Fold 4 IBS: 0.17521730364804317
Fold 5 IBS: 0.22130334953577135
[I 2024-04-14 13:02:20,397] Trial 30 finished with value: 0.19026497104109502 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 2, 'min_samples_leaf': 3, 'max_depth': 4, 'n_estimators': 256, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.9487209920968901, 'min_weight_fraction_leaf': 0.18097362848347806}. Best is trial 26 with value: 0.18292392529805948.
Fold 1 IBS: 0.1635854354097278
Fold 2 IBS: 0.21121780051762484
Fold 3 IBS: 0.16113974292115463
Fold 4 IBS: 0.1505740975965663
Fold 5 IBS: 0.22708547819250702
[I 2024-04-14 13:02:23,307] Trial 31 finished with value: 0.18272051092751612 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 6, 'min_samples_leaf': 2, 'max_depth': 6, 'n_estimators': 328, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.742

Fold 1 IBS: 0.16018787401037188
Fold 2 IBS: 0.21157577684673273
Fold 3 IBS: 0.16062873421710938
Fold 4 IBS: 0.14895364309241638
Fold 5 IBS: 0.22578089787314537
[I 2024-04-14 13:03:12,701] Trial 45 finished with value: 0.18142538520795515 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 8, 'min_samples_leaf': 2, 'max_depth': 3, 'n_estimators': 491, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.9583231565912955, 'min_weight_fraction_leaf': 0.0029107293558666867}. Best is trial 45 with value: 0.18142538520795515.
Fold 1 IBS: 0.1677734556334294
Fold 2 IBS: 0.20073298892392583
Fold 3 IBS: 0.16790665630340973
Fold 4 IBS: 0.16323803409498044
Fold 5 IBS: 0.21692306693145388
[I 2024-04-14 13:03:17,065] Trial 46 finished with value: 0.1833148403774399 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 9, 'min_samples_leaf': 2, 'max_depth': 3, 'n_estimators': 492, 'oob_score': True, 'warm_start': False, 'max_features': 0.1, 'max_samples': 0.9

Fold 1 IBS: 0.16230315465603623
Fold 2 IBS: 0.22256653426146097
Fold 3 IBS: 0.1640671332820358
Fold 4 IBS: 0.16540280962956921
Fold 5 IBS: 0.2288950034663518
[I 2024-04-14 13:04:08,419] Trial 60 finished with value: 0.1886469270590908 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 8, 'min_samples_leaf': 2, 'max_depth': 1, 'n_estimators': 74, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.36612707136592737, 'min_weight_fraction_leaf': 0.0019516995494000267}. Best is trial 59 with value: 0.18071425285640869.
Fold 1 IBS: 0.168038042421559
Fold 2 IBS: 0.2051256684585529
Fold 3 IBS: 0.16229862942852732
Fold 4 IBS: 0.15452249752369313
Fold 5 IBS: 0.22616041975575948
[I 2024-04-14 13:04:11,485] Trial 61 finished with value: 0.18322905151761837 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 7, 'min_samples_leaf': 1, 'max_depth': 3, 'n_estimators': 357, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.5223

Fold 1 IBS: 0.17451230590744263
Fold 2 IBS: 0.20100399290448753
Fold 3 IBS: 0.17156096463153092
Fold 4 IBS: 0.1718768459456622
Fold 5 IBS: 0.21682889436787772
[I 2024-04-14 13:37:30,480] Trial 75 finished with value: 0.1871566007514002 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 5, 'min_samples_leaf': 2, 'max_depth': 15, 'n_estimators': 391, 'oob_score': True, 'warm_start': False, 'max_features': 1, 'max_samples': 0.9444744925227744, 'min_weight_fraction_leaf': 0.07181205233641544}. Best is trial 59 with value: 0.18071425285640869.
Fold 1 IBS: 0.16363885977998363
Fold 2 IBS: 0.21080703897734426
Fold 3 IBS: 0.1638208483918266
Fold 4 IBS: 0.15977599581486032
Fold 5 IBS: 0.22548859529714996
[I 2024-04-14 13:37:33,597] Trial 76 finished with value: 0.18470626765223294 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 3, 'min_samples_leaf': 3, 'max_depth': 3, 'n_estimators': 499, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.970609

Fold 1 IBS: 0.18505726472199197
Fold 2 IBS: 0.20422882801752185
Fold 3 IBS: 0.18218146544483949
Fold 4 IBS: 0.197363394678087
Fold 5 IBS: 0.21363202834995396
[I 2024-04-14 13:38:11,522] Trial 90 finished with value: 0.1964925962424789 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 6, 'min_samples_leaf': 1, 'max_depth': 6, 'n_estimators': 333, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.8685241268999454, 'min_weight_fraction_leaf': 0.36113729645899395}. Best is trial 59 with value: 0.18071425285640869.
Fold 1 IBS: 0.16232027984914077
Fold 2 IBS: 0.21039259861689338
Fold 3 IBS: 0.16262694408072945
Fold 4 IBS: 0.15691427656551898
Fold 5 IBS: 0.2239182334907605
[I 2024-04-14 13:47:06,825] Trial 91 finished with value: 0.18323446652060862 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 5, 'min_samples_leaf': 2, 'max_depth': 2, 'n_estimators': 400, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.98

In [87]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [88]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.818
train_ibs:  0.181


#### Test

In [89]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [90]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=17, max_features=1, max_leaf_nodes=11,
                   max_samples=0.899094401409131, min_samples_leaf=1,
                   min_weight_fraction_leaf=0.00386845300436138,
                   n_estimators=380, oob_score=True, random_state=123,
                   warm_start=True)

C-index score: 0.607


ExtraSurvivalTrees(max_depth=3, max_features='auto', max_leaf_nodes=7,
                   max_samples=0.7977445310979885, min_samples_leaf=1,
                   min_samples_split=12,
                   min_weight_fraction_leaf=5.246908224487007e-05,
                   n_estimators=348, oob_score=True, random_state=123)

IBS: 0.21


In [91]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [92]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 13:47:27,975] A new study created in memory with name: no-name-cfe6785d-477b-4ca7-890b-c6fb8b39a8e0


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 14:11:55,340] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 14:28:25,243] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 14:37:57,820] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.7322716318897614.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 14:38:26,369] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 14:43:49,450] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.7565257046780437, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.3433523077110169, 'n_estimators': 318, 'criterion': 'squared_error', 'ccp_alpha': 2.063559212730723, 'min_weight_fraction_leaf': 0.25401273903421573, 'max_features': 'auto', 'min_impurity_decrease': 5.773435946665558e-07, 'validation_fraction': 0.8062268184400477, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 9 with value: 0.7322716318897614.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.6004464285714286
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.6103286384976526
[I 2024-04-14 14:44:29,457] Trial 26 finished with value: 0.5421550134138162 and parameters: {'subsample': 0.8922404482621683, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.2527000999648632, 'n_estimators': 446,

...

Fold 2 IBS: 0.22157791718667935
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-14 15:31:24,590] Trial 55 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.9027683411994925, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.13472529918097312, 'n_estimators': 381, 'criterion': 'squared_error', 'ccp_alpha': 1.3381569877935875, 'min_weight_fraction_leaf': 0.36938177041618503, 'max_features': 'auto', 'min_impurity_decrease': 1.1016843774656315e-07, 'validation_fraction': 0.898549324711475, 'min_samples_split': 3, 'max_leaf_nodes': 12, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 53 with value: 0.21504561735962566.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609582
[I 2024-04-14 15:31:59,449] Trial 56 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.83247895380525

Fold 2 IBS: 0.22157791718667935
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609582
[I 2024-04-14 15:40:48,935] Trial 66 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.8639813037885321, 'learning_rate': 0.010366561182703663, 'dropout_rate': 0.2054023171521986, 'n_estimators': 500, 'criterion': 'squared_error', 'ccp_alpha': 1.0590752315578722, 'min_weight_fraction_leaf': 0.1974211678184809, 'max_features': 'log2', 'min_impurity_decrease': 4.836772048236878e-06, 'validation_fraction': 0.9971715713668047, 'min_samples_split': 18, 'max_leaf_nodes': 14, 'min_samples_leaf': 11, 'max_depth': 4}. Best is trial 53 with value: 0.21504561735962566.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609557
[I 2024-04-14 15:41:41,858] Trial 67 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.92471115127087

Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667935
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609582
[I 2024-04-14 15:48:53,735] Trial 77 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.9392590368987991, 'learning_rate': 0.011132229721027662, 'dropout_rate': 0.25203573700169807, 'n_estimators': 441, 'criterion': 'squared_error', 'ccp_alpha': 0.33381590515318865, 'min_weight_fraction_leaf': 0.13051249394026804, 'max_features': 1, 'min_impurity_decrease': 7.70123021698274e-05, 'validation_fraction': 0.994934454010795, 'min_samples_split': 18, 'max_leaf_nodes': 13, 'min_samples_leaf': 18, 'max_depth': 3}. Best is trial 53 with value: 0.21504561735962566.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667935
Fold 3 IBS: 0.20453594732018132
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609582
[I 2024-04-14 15:49:43,881] Trial 78 finished with value: 0.21659054862241586 and parameters: {'su

Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609574
[I 2024-04-14 15:55:13,103] Trial 88 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.8335693537780458, 'learning_rate': 0.025896998029808806, 'dropout_rate': 0.2592233887917012, 'n_estimators': 416, 'criterion': 'squared_error', 'ccp_alpha': 1.998777407817298, 'min_weight_fraction_leaf': 0.06882085091143297, 'max_features': 'log2', 'min_impurity_decrease': 2.1900682995121406e-06, 'validation_fraction': 0.9382598277453086, 'min_samples_split': 19, 'max_leaf_nodes': 18, 'min_samples_leaf': 18, 'max_depth': 3}. Best is trial 85 with value: 0.2140003401212934.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609566
[I 2024-04-14 15:55:23,912] Trial 89 finished with value: 0.21659054862241583 and parameters: 

Fold 1 IBS: 0.21256519342739535
Fold 2 IBS: 0.22132501050202244
Fold 3 IBS: 0.2031414835960969
Fold 4 IBS: 0.22299974010438342
Fold 5 IBS: 0.21780142809241373
[I 2024-04-14 16:02:05,842] Trial 99 finished with value: 0.21556657114446237 and parameters: {'subsample': 0.9762156523558686, 'learning_rate': 0.01377474153403631, 'dropout_rate': 0.2717132298988158, 'n_estimators': 463, 'criterion': 'squared_error', 'ccp_alpha': 0.004394120129034387, 'min_weight_fraction_leaf': 0.28881659395589254, 'max_features': 'auto', 'min_impurity_decrease': 1.3534409368328512e-06, 'validation_fraction': 0.6748895535913024, 'min_samples_split': 19, 'max_leaf_nodes': 12, 'min_samples_leaf': 13, 'max_depth': 17}. Best is trial 85 with value: 0.2140003401212934.


* Best trial for IBS: 
 FrozenTrial(number=85, state=TrialState.COMPLETE, values=[0.2140003401212934], datetime_start=datetime.datetime(2024, 4, 14, 15, 53, 50, 224687), datetime_complete=datetime.datetime(2024, 4, 14, 15, 54, 7, 85282), params={'s

In [93]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [94]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.75
train_ibs:  0.214


#### Test

In [95]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [96]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.03063070291051248,
                                 criterion='squared_error',
                                 dropout_rate=0.2576884847115747,
                                 learning_rate=0.007359366951045268,
                                 max_depth=16, max_features=1,
                                 max_leaf_nodes=16,
                                 min_impurity_decrease=1.3903906488794697e-07,
                                 min_samples_leaf=14, min_samples_split=20,
                                 min_weight_fraction_leaf=0.38180626899357545,
                                 n_estimators=96, random_state=123,
                                 subsample=0.8938290428827321,
                                 validation_fraction=0.9577535215137098)

C-index score: 0.604


GradientBoostingSurvivalAnalysis(ccp_alpha=0.011241474018604543,
                                 criterion='squared_error',
                                 dropout_rate=0.10391090561180716,
                                 learning_rate=0.014554866476952134,
                                 max_depth=1, max_features='auto',
                                 max_leaf_nodes=18,
                                 min_impurity_decrease=1.8697047323397577e-06,
                                 min_samples_leaf=19, min_samples_split=20,
                                 min_weight_fraction_leaf=0.12733224205170357,
                                 n_estimators=265, random_state=123,
                                 subsample=0.8559905830599279,
                                 validation_fraction=0.9999275626013308)

IBS: 0.218


In [97]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [98]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 16:02:11,365] A new study created in memory with name: no-name-1f363243-09ce-4a93-996b-c114b770d2bd


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6190476190476191
Fold 2 C-index: 0.609375
Fold 3 C-index: 0.7450980392156863
Fold 4 C-index: 0.7848101265822784
Fold 5 C-index: 0.6103286384976526
[I 2024-04-14 16:02:12,195] Trial 0 finished with value: 0.6737318846686473 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6737318846686473.
Fold 1 C-index: 0.6190476190476191
Fold 2 C-index: 0.609375
Fold 3 C-index: 0.7916666666666666
Fold 4 C-index: 0.7848101265822784
Fold 5 C-index: 0.6103286384976526
[I 2024-04-14 16:02:18,149] Trial 1 finished with value: 0.6830456101588432 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 1 with value: 0.6830456101588432.
Fold 1 C-index: 0.6363636363636364
Fold 2 C-index: 0.609375
Fold 3 C-index: 0.7916666666666666
Fold 4 C-index: 0.7848101265822784


Fold 1 C-index: 0.6666666666666666
Fold 2 C-index: 0.6919642857142857
Fold 3 C-index: 0.7916666666666666
Fold 4 C-index: 0.8227848101265823
Fold 5 C-index: 0.6126760563380281
[I 2024-04-14 16:03:09,247] Trial 19 finished with value: 0.7171516971024459 and parameters: {'subsample': 0.34314044045637715, 'dropout_rate': 0.981979648714702, 'n_estimators': 426, 'learning_rate': 0.09857240282031608}. Best is trial 12 with value: 0.7397589407811294.
Fold 1 C-index: 0.6753246753246753
Fold 2 C-index: 0.6964285714285714
Fold 3 C-index: 0.8186274509803921
Fold 4 C-index: 0.8227848101265823
Fold 5 C-index: 0.6126760563380281
[I 2024-04-14 16:03:09,899] Trial 20 finished with value: 0.7251683128396499 and parameters: {'subsample': 0.2628688080573812, 'dropout_rate': 0.5777838682216713, 'n_estimators': 112, 'learning_rate': 0.0794193638388259}. Best is trial 12 with value: 0.7397589407811294.
Fold 1 C-index: 0.6926406926406926
Fold 2 C-index: 0.6919642857142857
Fold 3 C-index: 0.8088235294117647
Fo

Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.7053571428571429
Fold 3 C-index: 0.8186274509803921
Fold 4 C-index: 0.8565400843881856
Fold 5 C-index: 0.647887323943662
[I 2024-04-14 16:04:03,208] Trial 38 finished with value: 0.7433447380962142 and parameters: {'subsample': 0.1504722419513361, 'dropout_rate': 0.13272164755980653, 'n_estimators': 209, 'learning_rate': 0.060145822351412026}. Best is trial 38 with value: 0.7433447380962142.
Fold 1 C-index: 0.658008658008658
Fold 2 C-index: 0.6741071428571429
Fold 3 C-index: 0.7916666666666666
Fold 4 C-index: 0.8523206751054853
Fold 5 C-index: 0.6126760563380281
[I 2024-04-14 16:04:04,610] Trial 39 finished with value: 0.7177558397951962 and parameters: {'subsample': 0.25449269996477825, 'dropout_rate': 0.4352656711347185, 'n_estimators': 159, 'learning_rate': 0.01715036845928685}. Best is trial 38 with value: 0.7433447380962142.
Fold 1 C-index: 0.6796536796536796
Fold 2 C-index: 0.6875
Fold 3 C-index: 0.8186274509803921
Fold 4 C-ind

...

Fold 4 IBS: 0.2022095506401677
Fold 5 IBS: 0.2127622739815413
[I 2024-04-14 16:08:21,965] Trial 19 finished with value: 0.19797104011955632 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.9930703343219778, 'n_estimators': 51, 'learning_rate': 0.04096883532946153}. Best is trial 19 with value: 0.19797104011955632.
Fold 1 IBS: 0.20068725004069762
Fold 2 IBS: 0.20471613577420322
Fold 3 IBS: 0.1822100971019276
Fold 4 IBS: 0.2037658040680384
Fold 5 IBS: 0.21298081915188694
[I 2024-04-14 16:08:22,205] Trial 20 finished with value: 0.20087202122735076 and parameters: {'subsample': 0.1435040576337751, 'dropout_rate': 0.7933084651006226, 'n_estimators': 43, 'learning_rate': 0.04127909023805986}. Best is trial 19 with value: 0.19797104011955632.
Fold 1 IBS: 0.19923087328677447
Fold 2 IBS: 0.20365798928368803
Fold 3 IBS: 0.1877487574780825
Fold 4 IBS: 0.20969793119821276
Fold 5 IBS: 0.21303933620208118
[I 2024-04-14 16:08:22,456] Trial 21 finished with value: 0.20267497748976

Fold 5 IBS: 0.26662175943776306
[I 2024-04-14 16:08:32,174] Trial 38 finished with value: 0.25295308188258814 and parameters: {'subsample': 0.35903417688251527, 'dropout_rate': 0.13272164755980653, 'n_estimators': 193, 'learning_rate': 0.07288612365177065}. Best is trial 32 with value: 0.19604144024771647.
Fold 1 IBS: 0.1972909823247562
Fold 2 IBS: 0.20320053346101893
Fold 3 IBS: 0.17708674952568834
Fold 4 IBS: 0.20372925950742424
Fold 5 IBS: 0.2114049622527249
[I 2024-04-14 16:08:32,471] Trial 39 finished with value: 0.19854249741432253 and parameters: {'subsample': 0.20700382404468165, 'dropout_rate': 0.9399951930137062, 'n_estimators': 86, 'learning_rate': 0.026251810001924513}. Best is trial 32 with value: 0.19604144024771647.
Fold 1 IBS: 0.19417702817107751
Fold 2 IBS: 0.2078048226758091
Fold 3 IBS: 0.17035677829557183
Fold 4 IBS: 0.19793915275395146
Fold 5 IBS: 0.21558203755241076
[I 2024-04-14 16:08:33,076] Trial 40 finished with value: 0.19717196388976416 and parameters: {'subs

Fold 5 IBS: 0.213339381812701
[I 2024-04-14 16:08:49,118] Trial 57 finished with value: 0.19575381685541493 and parameters: {'subsample': 0.10313673325198738, 'dropout_rate': 0.421255149383966, 'n_estimators': 162, 'learning_rate': 0.018004642634063723}. Best is trial 57 with value: 0.19575381685541493.
Fold 1 IBS: 0.19280446652074382
Fold 2 IBS: 0.20352814494078572
Fold 3 IBS: 0.18000828642387096
Fold 4 IBS: 0.21218869787241948
Fold 5 IBS: 0.21078621095198513
[I 2024-04-14 16:08:49,521] Trial 58 finished with value: 0.19986316134196103 and parameters: {'subsample': 0.7591484265070766, 'dropout_rate': 0.17404595544613377, 'n_estimators': 108, 'learning_rate': 0.018185519971222298}. Best is trial 57 with value: 0.19575381685541493.
Fold 1 IBS: 0.19554473539337386
Fold 2 IBS: 0.20398169137465574
Fold 3 IBS: 0.16932903909533784
Fold 4 IBS: 0.19333352166708745
Fold 5 IBS: 0.2177867063000703
[I 2024-04-14 16:08:49,850] Trial 59 finished with value: 0.19599513876610503 and parameters: {'subs

Fold 3 IBS: 0.1925557724764544
Fold 4 IBS: 0.2127816245380913
Fold 5 IBS: 0.2134268175780662
[I 2024-04-14 16:08:55,574] Trial 77 finished with value: 0.2068432884673709 and parameters: {'subsample': 0.2240545714205724, 'dropout_rate': 0.2677960442541683, 'n_estimators': 14, 'learning_rate': 0.062339761242765926}. Best is trial 74 with value: 0.1951760581180958.
Fold 1 IBS: 0.19688995298276135
Fold 2 IBS: 0.21142148860933466
Fold 3 IBS: 0.16640432906604044
Fold 4 IBS: 0.20699514299097024
Fold 5 IBS: 0.22400216335201178
[I 2024-04-14 16:08:55,989] Trial 78 finished with value: 0.20114261540022368 and parameters: {'subsample': 0.15412816920461153, 'dropout_rate': 0.1645556222499458, 'n_estimators': 102, 'learning_rate': 0.05561449846981878}. Best is trial 74 with value: 0.1951760581180958.
Fold 1 IBS: 0.19250802046649604
Fold 2 IBS: 0.19996534210019845
Fold 3 IBS: 0.17285018388546095
Fold 4 IBS: 0.19440265323113975
Fold 5 IBS: 0.21939539321623278
[I 2024-04-14 16:08:56,316] Trial 79 fini

Fold 4 IBS: 0.20988660978839166
Fold 5 IBS: 0.21456843353041302
[I 2024-04-14 16:09:05,489] Trial 96 finished with value: 0.2040135416142545 and parameters: {'subsample': 0.14090987604874106, 'dropout_rate': 0.2471454520467835, 'n_estimators': 22, 'learning_rate': 0.05873889729527442}. Best is trial 91 with value: 0.19464063603113557.
Fold 1 IBS: 0.19110700040821932
Fold 2 IBS: 0.2020852390234278
Fold 3 IBS: 0.17881544416715775
Fold 4 IBS: 0.203756488490083
Fold 5 IBS: 0.21163899451939966
[I 2024-04-14 16:09:05,663] Trial 97 finished with value: 0.1974806333216575 and parameters: {'subsample': 0.19182777199313375, 'dropout_rate': 0.19407730384524952, 'n_estimators': 35, 'learning_rate': 0.06423903598156665}. Best is trial 91 with value: 0.19464063603113557.
Fold 1 IBS: 0.21294104213529513
Fold 2 IBS: 0.22063662299714767
Fold 3 IBS: 0.2036617330808848
Fold 4 IBS: 0.22387059770553963
Fold 5 IBS: 0.2177490125412687
[I 2024-04-14 16:09:05,786] Trial 98 finished with value: 0.21577180169202

In [99]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [100]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.743
train_ibs:  0.195


#### Test

In [101]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [102]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.13272164755980653,
                                              learning_rate=0.060145822351412026,
                                              n_estimators=209,
                                              random_state=123,
                                              subsample=0.1504722419513361)

C-index score: 0.652


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.23080692510197112,
                                              learning_rate=0.048810527071028224,
                                              n_estimators=57, random_state=123,
                                              subsample=0.1526734943707138)

IBS: 0.206


In [103]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [104]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.821,1.0
ExtraSurvivalTrees,0.818,2.0
CoxLasso,0.762,3.5
CoxElastic,0.762,3.5
CoxPH,0.758,5.0
GradientBoosting,0.750,6.0
ComponentwiseGradientBoosting,0.743,7.0
CoxRidge,0.679,8.0


In [105]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
Randomsurvivalforest,0.181,1.5
ExtraSurvivalTrees,0.181,1.5
CoxPH,0.182,4.0
CoxLasso,0.182,4.0
CoxElastic,0.182,4.0
ComponentwiseGradientBoosting,0.195,6.0
GradientBoosting,0.214,7.0
CoxRidge,0.217,8.0


In [106]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
Randomsurvivalforest,0.683,1.0
ComponentwiseGradientBoosting,0.652,2.0
CoxRidge,0.647,3.0
CoxLasso,0.623,4.5
CoxElastic,0.623,4.5
CoxPH,0.621,6.0
ExtraSurvivalTrees,0.607,7.0
GradientBoosting,0.604,8.0


In [107]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
Randomsurvivalforest,0.201,1.0
ComponentwiseGradientBoosting,0.206,2.0
ExtraSurvivalTrees,0.210,3.0
GradientBoosting,0.218,4.0
CoxRidge,0.221,5.0
CoxElastic,0.257,6.0
CoxLasso,0.258,7.0
CoxPH,0.261,8.0


In [108]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d1/os/standard/plsr/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

modified_file_names = ['d1_os_standard_plsr_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [109]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-14
